# 22 GSE65858_clinical_adjustment_ROUND2

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import warnings
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning

In [ ]:
scores = pd.read_csv("VERIFIED_CORRECT_scores.csv").rename(columns={"GSM_Sample_ID": "Sample"})
clinical = pd.read_csv("gse65858_clinical_variables.csv")
df = scores.merge(clinical, on="Sample", suffixes=("", "_clin"))
print("Merged:", df.shape)

In [ ]:
from lifelines import CoxPHFitter as _CPH
expected = {
    "Score_lambda_min": (1.81, 0.016), "Score_lambda_1se": (2.65, 0.024),
    "Score_Toustrup": (1.27, 0.179), "Score_Eustace": (0.99, 0.94),
    "Score_Lendahl": (1.42, 0.172), "Score_RSI": (3.18, 0.245),
}
for col, (exp_hr, exp_p) in expected.items():
    d = df[["OS_months","Event",col]].dropna()
    m = _CPH(); m.fit(d, duration_col="OS_months", event_col="Event")
    s = m.summary.loc[col]
    hr, p = np.exp(s["coef"]), s["p"]
    status = "OK" if abs(hr-exp_hr) < 0.02 and abs(p-exp_p) < 0.005 else "MISMATCH"
    print(f"{col}: HR={hr:.3f} (expected {exp_hr}), p={p:.3f} (expected {exp_p}) -- {status}")

In [ ]:
def stage_group(s):
    if pd.isna(s): return np.nan
    s = str(s).strip()
    if s in ["I", "II"]: return "I-II"
    elif s == "III": return "III"
    elif s in ["IVA", "IVB", "IVC"]: return "IV"
    return np.nan

df["stage_group"] = df["uicc_stage"].apply(stage_group)
df["age_num"] = pd.to_numeric(df["age"], errors="coerce")

main_vars = ["age_num", "stage_group", "tumor_site", "hpv_dna", "smoking"]
main_cohort = df.dropna(subset=main_vars + ["OS_months", "Event"]).copy()
print(f"MAIN cohort (simple HPV): N={len(main_cohort)}, events={int(main_cohort['Event'].sum())}")

all_vars = main_vars + ["hpv16_dna_rna"]
sens_cohort = df.dropna(subset=all_vars + ["OS_months", "Event"]).copy()
print(f"SENSITIVITY cohort (both HPV specs): N={len(sens_cohort)}, events={int(sens_cohort['Event'].sum())}")

In [ ]:
def encode(data, hpv_col, hpv_prefix, hpv_ref_contains):
    d = data.copy()
    d = pd.get_dummies(d, columns=["stage_group"], prefix="stage", drop_first=False)
    d = d.drop(columns=["stage_I-II"])
    d = pd.get_dummies(d, columns=["tumor_site"], prefix="site", drop_first=False)
    site_ref = [c for c in d.columns if c.startswith("site_") and "Oropharynx" in c]
    d = d.drop(columns=site_ref)
    d = pd.get_dummies(d, columns=[hpv_col], prefix=hpv_prefix, drop_first=False)
    hpv_ref = [c for c in d.columns if c.startswith(hpv_prefix + "_") and hpv_ref_contains in c]
    d = d.drop(columns=hpv_ref)
    d = pd.get_dummies(d, columns=["smoking"], prefix="smoking", drop_first=True)
    return d

def fit_model(data, covariate_prefixes, penalizer, label, score_col=None):
    d = data.copy()
    covariate_cols = [c for c in d.columns if c.startswith(covariate_prefixes)]
    if score_col:
        score_sd = d[score_col].std()
        d[f"{score_col}_perSD"] = d[score_col] / score_sd
        model_cols = ["OS_months", "Event", f"{score_col}_perSD", "age_num"] + covariate_cols
    else:
        model_cols = ["OS_months", "Event", "age_num"] + covariate_cols

    dm = d[model_cols].dropna().copy()
    for c in covariate_cols + (["age_num"] if not score_col else [f"{score_col}_perSD", "age_num"]):
        dm[c] = dm[c].astype(float)

    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(dm, duration_col="OS_months", event_col="Event")
        conv_warnings = [str(x.message) for x in w if issubclass(x.category, ConvergenceWarning)]

    cidx = cph.concordance_index_
    result = {"label": label, "N": len(dm), "events": int(dm["Event"].sum()), "C_index": cidx, "n_warnings": len(conv_warnings)}
    print(f"{label}: N={result['N']}, events={result['events']}, warnings={result['n_warnings']}", end="")
    if score_col:
        summ = cph.summary.loc[f"{score_col}_perSD"]
        hr, lo, hi, pval = np.exp(summ["coef"]), np.exp(summ["coef lower 95%"]), np.exp(summ["coef upper 95%"]), summ["p"]
        result.update({"HR_per_SD": hr, "CI_low": lo, "CI_high": hi, "p": pval})
        print(f", HR/SD={hr:.4f} ({lo:.4f}-{hi:.4f}), p={pval:.5f}, C-index={cidx:.4f}")
        cph.check_assumptions(dm, p_value_threshold=0.05, show_plots=False)
    else:
        print(f", C-index={cidx:.4f} (clinical factors only)")
    return result

In [ ]:
main_enc = encode(main_cohort, "hpv_dna", "hpv", "Negative")
main_r1 = fit_model(main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "MAIN -- lambda.min + clinical", "Score_lambda_min")

In [ ]:
main_r2 = fit_model(main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "MAIN -- lambda.1se + clinical", "Score_lambda_1se")

In [ ]:
main_r3 = fit_model(main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "MAIN -- CLINICAL ONLY")

In [ ]:
sens_main_enc = encode(sens_cohort, "hpv_dna", "hpv", "Negative")
sens_r1 = fit_model(sens_main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "SENS (simple HPV) -- lambda.min + clinical", "Score_lambda_min")
sens_r2 = fit_model(sens_main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "SENS (simple HPV) -- lambda.1se + clinical", "Score_lambda_1se")
sens_r3 = fit_model(sens_main_enc, ("stage_","site_","hpv_","smoking_"), 0.0, "SENS (simple HPV) -- CLINICAL ONLY")

In [ ]:
sens_detail_enc = encode(sens_cohort, "hpv16_dna_rna", "hpvdetail", "DNA-")
sens_r4 = fit_model(sens_detail_enc, ("stage_","site_","hpvdetail_","smoking_"), 0.0, "SENS (detailed HPV) -- lambda.min + clinical", "Score_lambda_min")
sens_r5 = fit_model(sens_detail_enc, ("stage_","site_","hpvdetail_","smoking_"), 0.0, "SENS (detailed HPV) -- lambda.1se + clinical", "Score_lambda_1se")
sens_r6 = fit_model(sens_detail_enc, ("stage_","site_","hpvdetail_","smoking_"), 0.0, "SENS (detailed HPV) -- CLINICAL ONLY")

In [ ]:
print(f"MAIN, simple HPV: lambda.min {main_r1['C_index']:.4f} vs clinical-only {main_r3['C_index']:.4f} = +{main_r1['C_index']-main_r3['C_index']:.4f}")
print(f"MAIN, simple HPV: lambda.1se {main_r2['C_index']:.4f} vs clinical-only {main_r3['C_index']:.4f} = +{main_r2['C_index']-main_r3['C_index']:.4f}")
print(f"SENS, simple HPV: lambda.min {sens_r1['C_index']:.4f} vs clinical-only {sens_r3['C_index']:.4f} = +{sens_r1['C_index']-sens_r3['C_index']:.4f}")
print(f"SENS, simple HPV: lambda.1se {sens_r2['C_index']:.4f} vs clinical-only {sens_r3['C_index']:.4f} = +{sens_r2['C_index']-sens_r3['C_index']:.4f}")
print(f"SENS, detailed HPV: lambda.min {sens_r4['C_index']:.4f} vs clinical-only {sens_r6['C_index']:.4f} = +{sens_r4['C_index']-sens_r6['C_index']:.4f}")
print(f"SENS, detailed HPV: lambda.1se {sens_r5['C_index']:.4f} vs clinical-only {sens_r6['C_index']:.4f} = +{sens_r5['C_index']-sens_r6['C_index']:.4f}")

all_results = [main_r1, main_r2, main_r3, sens_r1, sens_r2, sens_r3, sens_r4, sens_r5, sens_r6]
pd.DataFrame(all_results).to_csv("round2_adjustment_summary.csv", index=False)